## 1. Setup and Data Loading

First, we'll import the necessary libraries and load the `ledger.csv` and `gateway.csv` files into pandas DataFrames. We'll also display basic information about each DataFrame to understand their structure and data types.

In [1]:
import pandas as pd
import json

# Load the datasets
ledger_df = pd.read_csv('ledger.csv')
gateway_df = pd.read_csv('gateway.csv')

print("--- Ledger DataFrame Info ---")
ledger_df.info()
print("\n--- Gateway DataFrame Info ---")
gateway_df.info()

print("\n--- Ledger Head ---")
display(ledger_df.head())
print("\n--- Gateway Head ---")
display(gateway_df.head())

--- Ledger DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    10 non-null     object 
 1   transaction_date  10 non-null     object 
 2   merchant_id       10 non-null     object 
 3   amount_usd        10 non-null     float64
 4   status            10 non-null     object 
 5   payment_method    10 non-null     object 
dtypes: float64(1), object(5)
memory usage: 612.0+ bytes

--- Gateway DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    9 non-null      object 
 1   transaction_date  9 non-null      object 
 2   merchant_id       9 non-null      object 
 3   amount_usd        9 non-null      float64
 4   status           

,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,850.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R004,2026-03-02,M003,2100.0,success,Card
4,R005,2026-03-03,M004,7200.0,success,Card



--- Gateway Head ---


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,900.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R005,2026-03-03,M004,7200.0,failed,Card
4,R006,2026-03-03,M002,950.0,success,UPI


## 2. Initial Data Validation (Duplicates and Nulls)

Before proceeding with reconciliation, it's crucial to check for duplicate `transaction_id` values and any nulls in critical columns to ensure data integrity.

In [2]:
print("--- Duplicates in Ledger (transaction_id) ---")
duplicate_ledger_ids = ledger_df[ledger_df.duplicated(subset=['transaction_id'], keep=False)]
if not duplicate_ledger_ids.empty:
    display(duplicate_ledger_ids)
else:
    print("No duplicate transaction_ids found in ledger.")

print("\n--- Nulls in Ledger ---")
nulls_in_ledger = ledger_df.isnull().sum()
print(nulls_in_ledger[nulls_in_ledger > 0])

print("\n--- Duplicates in Gateway (transaction_id) ---")
duplicate_gateway_ids = gateway_df[gateway_df.duplicated(subset=['transaction_id'], keep=False)]
if not duplicate_gateway_ids.empty:
    display(duplicate_gateway_ids)
else:
    print("No duplicate transaction_ids found in gateway.")

print("\n--- Nulls in Gateway ---")
nulls_in_gateway = gateway_df.isnull().sum()
print(nulls_in_gateway[nulls_in_gateway > 0])

--- Duplicates in Ledger (transaction_id) ---
No duplicate transaction_ids found in ledger.

--- Nulls in Ledger ---
Series([], dtype: int64)

--- Duplicates in Gateway (transaction_id) ---
No duplicate transaction_ids found in gateway.

--- Nulls in Gateway ---
Series([], dtype: int64)


## 3. Prepare Data for Merging and Reconciliation

To ensure accurate reconciliation, we need to standardize column names and data types, especially for `transaction_date` and `amount_usd`. We'll also fill any crucial missing values before the merge operation.

In [3]:
# Ensure 'transaction_id' is consistent (e.g., string type for merging reliability)
ledger_df['transaction_id'] = ledger_df['transaction_id'].astype(str)
gateway_df['transaction_id'] = gateway_df['transaction_id'].astype(str)

# Convert date columns to datetime objects for consistency
# Using errors='coerce' will turn unparseable dates into NaT (Not a Time)
ledger_df['transaction_date'] = pd.to_datetime(ledger_df['transaction_date'], errors='coerce')
gateway_df['transaction_date'] = pd.to_datetime(gateway_df['transaction_date'], errors='coerce')

# Convert amount columns to numeric, filling non-numeric with NaN, then with 0 as per rule
ledger_df['amount_usd'] = pd.to_numeric(ledger_df['amount_usd'], errors='coerce').fillna(0)
gateway_df['amount_usd'] = pd.to_numeric(gateway_df['amount_usd'], errors='coerce').fillna(0)

# Fill any remaining NaNs in 'status' and 'payment_method' with 'NA' for consistency
ledger_df['status'] = ledger_df['status'].fillna('NA')
gateway_df['status'] = gateway_df['status'].fillna('NA')

ledger_df['payment_method'] = ledger_df['payment_method'].fillna('NA')
gateway_df['payment_method'] = gateway_df['payment_method'].fillna('NA')

## 4. Perform Full Outer Join for Comprehensive Reconciliation

We will perform a full outer join on `transaction_id`. This merge will include all transactions from both the ledger and gateway, allowing us to identify records present in one system but not the other, as well as discrepancies in common transactions.

In [4]:
merged_df = pd.merge(ledger_df, gateway_df, on='transaction_id', how='outer',
                       suffixes=('_ledger', '_gateway'))

print("--- Merged DataFrame Head (Outer Join) ---")
display(merged_df.head())
print(f"Merged DataFrame Shape: {merged_df.shape}")

--- Merged DataFrame Head (Outer Join) ---


,transaction_id,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway
0,R001,2026-03-01,M001,1200.0,success,UPI,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,850.0,success,Card,2026-03-01,M002,900.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet,2026-03-02,M001,500.0,success,Wallet
3,R004,2026-03-02,M003,2100.0,success,Card,NaT,NaN,NaN,NaN,NaN
4,R005,2026-03-03,M004,7200.0,success,Card,2026-03-03,M004,7200.0,failed,Card


Merged DataFrame Shape: (11, 11)


## 5. Identify and Classify Discrepancies

Based on the merged DataFrame, we'll identify and classify the four types of discrepancies:

*   **Missing in Gateway:** Transactions present in the ledger but not in the gateway.
*   **Missing in Ledger:** Transactions present in the gateway but not in the ledger.
*   **Amount Mismatch:** Transactions present in both, but with differing `amount_usd`.
*   **Status Mismatch:** Transactions present in both, but with differing `status`.

We will create separate DataFrames for each type of discrepancy and save them to CSV files as required.

In [5]:
# 5.1. Identify Records Missing in Gateway
missing_in_gateway = merged_df[merged_df['transaction_date_gateway'].isnull() & merged_df['transaction_date_ledger'].notnull()].copy()
if not missing_in_gateway.empty:
    print(f"Found {len(missing_in_gateway)} records in ledger missing from gateway:")
    display(missing_in_gateway[['transaction_id', 'transaction_date_ledger', 'amount_usd_ledger']])
    missing_in_gateway.to_csv('missing_in_gateway.csv', index=False)
    print("Saved missing_in_gateway.csv")
else:
    print("No records found in ledger missing from gateway.")

# 5.2. Identify Records Missing in Ledger
missing_in_ledger = merged_df[merged_df['transaction_date_ledger'].isnull() & merged_df['transaction_date_gateway'].notnull()].copy()
if not missing_in_ledger.empty:
    print(f"\nFound {len(missing_in_ledger)} records in gateway missing from ledger:")
    display(missing_in_ledger[['transaction_id', 'transaction_date_gateway', 'amount_usd_gateway']])
    missing_in_ledger.to_csv('missing_in_ledger.csv', index=False)
    print("Saved missing_in_ledger.csv")
else:
    print("\nNo records found in gateway missing from ledger.")

# Common transactions (present in both ledger and gateway)
common_transactions = merged_df[merged_df['transaction_date_ledger'].notnull() & merged_df['transaction_date_gateway'].notnull()].copy()

# 5.3. Identify Amount Mismatches
# Note: amounts were converted to numeric and NaNs filled with 0 earlier for safe comparison
amount_mismatches = common_transactions[common_transactions['amount_usd_ledger'] != common_transactions['amount_usd_gateway']].copy()
if not amount_mismatches.empty:
    print(f"\nFound {len(amount_mismatches)} records with amount mismatches:")
    display(amount_mismatches[['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway']])
    amount_mismatches.to_csv('amount_mismatches.csv', index=False)
    print("Saved amount_mismatches.csv")
else:
    print("\nNo amount mismatches found between ledger and gateway.")

# 5.4. Identify Status Mismatches
# Note: status were converted to string and NaNs filled with 'NA' earlier for safe comparison
status_mismatches = common_transactions[common_transactions['status_ledger'] != common_transactions['status_gateway']].copy()
if not status_mismatches.empty:
    print(f"\nFound {len(status_mismatches)} records with status mismatches:")
    display(status_mismatches[['transaction_id', 'status_ledger', 'status_gateway']])
    status_mismatches.to_csv('status_mismatches.csv', index=False)
    print("Saved status_mismatches.csv")
else:
    print("\nNo status mismatches found between ledger and gateway.")

Found 2 records in ledger missing from gateway:


,transaction_id,transaction_date_ledger,amount_usd_ledger
3,R004,2026-03-02,2100.0
9,R010,2026-03-05,2500.0


Saved missing_in_gateway.csv

Found 1 records in gateway missing from ledger:


,transaction_id,transaction_date_gateway,amount_usd_gateway
10,R011,2026-03-05,1800.0


Saved missing_in_ledger.csv

Found 2 records with amount mismatches:


,transaction_id,amount_usd_ledger,amount_usd_gateway
1,R002,850.0,900.0
7,R008,640.0,600.0


Saved amount_mismatches.csv

Found 1 records with status mismatches:


,transaction_id,status_ledger,status_gateway
4,R005,success,failed


Saved status_mismatches.csv


## 6. Build Final Reconciliation Report

This is the core of the reconciliation process. We will create a single `reconciliation_report_df` that includes all discrepancies with a `mismatch_type` column, adhering to the specified data structure and type rules for Looker Studio compatibility.

We will prioritize mismatch types in the following order:
1.  Missing in Ledger
2.  Missing in Gateway
3.  Amount Mismatch
4.  Status Mismatch

This ensures that each discrepancy is classified uniquely. Transactions with no identified mismatch will not be included in the final report.

In [8]:
# Initialize mismatch_type column with a default value (e.g., 'No Mismatch')
# This will be updated based on conditions
merged_df['mismatch_type'] = 'No Mismatch'

# Apply mismatch types based on defined rules and priority
# Priority 1: Missing in Ledger
merged_df.loc[merged_df['transaction_date_ledger'].isnull() & merged_df['transaction_date_gateway'].notnull(), 'mismatch_type'] = 'Missing in Ledger'

# Priority 2: Missing in Gateway
merged_df.loc[merged_df['transaction_date_gateway'].isnull() & merged_df['transaction_date_ledger'].notnull(), 'mismatch_type'] = 'Missing in Gateway'

# Priority 3: Amount Mismatch for common transactions (only apply if not already classified as missing)
common_and_not_missing_idx = (merged_df['transaction_date_ledger'].notnull()) & \
                             (merged_df['transaction_date_gateway'].notnull()) & \
                             (merged_df['amount_usd_ledger'] != merged_df['amount_usd_gateway'])
merged_df.loc[common_and_not_missing_idx, 'mismatch_type'] = 'Amount Mismatch'

# Priority 4: Status Mismatch for common transactions (only apply if not already classified as missing or amount mismatch)
common_and_no_amount_mismatch_idx = (merged_df['transaction_date_ledger'].notnull()) & \
                                    (merged_df['transaction_date_gateway'].notnull()) & \
                                    (merged_df['amount_usd_ledger'] == merged_df['amount_usd_gateway']) & \
                                    (merged_df['status_ledger'] != merged_df['status_gateway']) # No need to fillna here, already done in step 3.
merged_df.loc[common_and_no_amount_mismatch_idx, 'mismatch_type'] = 'Status Mismatch'

# Filter for only actual discrepancies
reconciliation_report_df = merged_df[merged_df['mismatch_type'] != 'No Mismatch'].copy()

# Combine transaction_date and merchant_id from ledger/gateway, prioritizing ledger if both exist
# The 'transaction_id' column already exists from the outer merge and is the combined key.
reconciliation_report_df['transaction_date'] = reconciliation_report_df['transaction_date_ledger'].fillna(reconciliation_report_df['transaction_date_gateway'])
reconciliation_report_df['merchant_id'] = reconciliation_report_df['merchant_id_ledger'].fillna(reconciliation_report_df['merchant_id_gateway'])

# Select and rename columns as per DATA STRUCTURE REQUIREMENTS
final_columns = [
    'transaction_id',
    'transaction_date',
    'merchant_id',
    'amount_usd_ledger',
    'amount_usd_gateway',
    'status_ledger',
    'status_gateway',
    'mismatch_type'
]

reconciliation_report_df = reconciliation_report_df[final_columns].copy()

reconciliation_report_df.rename(columns={
    'amount_usd_ledger': 'ledger_amount',
    'amount_usd_gateway': 'gateway_amount',
    'status_ledger': 'ledger_status',
    'status_gateway': 'gateway_status'
}, inplace=True)

# Apply DATA TYPE RULES:
# ledger_amount and gateway_amount MUST be numeric (no strings) -> Already handled by .fillna(0) in step 3
# Fill missing numeric values with 0 -> Already handled by .fillna(0) in step 3 for amount_usd
# Fill missing status values with 'NA' -> Already handled by .fillna('NA') in step 3 for status

# For `transaction_date` and `merchant_id` in the final report, if one side was missing,
# the corresponding combined column might still have NaN/NaT. Convert to appropriate types
# and fill for Looker Studio compatibility.
reconciliation_report_df['transaction_date'] = reconciliation_report_df['transaction_date'].dt.strftime('%Y-%m-%d').fillna('NA') # Convert NaT to 'NA' string
reconciliation_report_df['merchant_id'] = reconciliation_report_df['merchant_id'].fillna('NA').astype(str) # Fill NaN and ensure string type


if not reconciliation_report_df.empty:
    print(f"Final reconciliation report generated with {len(reconciliation_report_df)} discrepancies:")
    display(reconciliation_report_df)
    reconciliation_report_df.to_csv('reconciliation_report.csv', index=False)
    print("Saved reconciliation_report.csv")
else:
    print("No discrepancies found in the reconciliation process.")

Final reconciliation report generated with 6 discrepancies:


,transaction_id,transaction_date,merchant_id,ledger_amount,gateway_amount,ledger_status,gateway_status,mismatch_type
1,R002,2026-03-01,M002,850.0,900.0,success,success,Amount Mismatch
3,R004,2026-03-02,M003,2100.0,NaN,success,NaN,Missing in Gateway
4,R005,2026-03-03,M004,7200.0,7200.0,success,failed,Status Mismatch
7,R008,2026-03-04,M001,640.0,600.0,success,success,Amount Mismatch
9,R010,2026-03-05,M004,2500.0,NaN,success,NaN,Missing in Gateway
10,R011,2026-03-05,M003,NaN,1800.0,NaN,success,Missing in Ledger


Saved reconciliation_report.csv


## 7. Generate Summary Metrics

We will generate a summary of all identified discrepancies and save it as a JSON file.

In [7]:
summary_metrics = {
    'total_ledger_transactions': len(ledger_df),
    'total_gateway_transactions': len(gateway_df),
    'missing_in_gateway_count': len(missing_in_gateway),
    'missing_in_ledger_count': len(missing_in_ledger),
    'amount_mismatches_count': len(amount_mismatches),
    'status_mismatches_count': len(status_mismatches),
    'total_discrepancies_in_report': len(reconciliation_report_df)
}

print("\n--- Summary Metrics ---")
print(json.dumps(summary_metrics, indent=4))

with open('summary_metrics.json', 'w') as f:
    json.dump(summary_metrics, f, indent=4)
print("Saved summary_metrics.json")


--- Summary Metrics ---
{
    "total_ledger_transactions": 10,
    "total_gateway_transactions": 9,
    "missing_in_gateway_count": 2,
    "missing_in_ledger_count": 1,
    "amount_mismatches_count": 2,
    "status_mismatches_count": 1,
    "total_discrepancies_in_report": 6
}
Saved summary_metrics.json


## 8. Explanation of Steps, Sample Output, and Looker Studio Tips

### Explanation of Each Step:

1.  **Setup and Data Loading**: Imports `pandas` for data manipulation and `json` for saving summary metrics. It loads `ledger.csv` and `gateway.csv` into DataFrames and provides an initial overview using `.info()` and `.head()`.
2.  **Initial Data Validation**: Checks for duplicate `transaction_id` entries and lists columns with null values in both DataFrames. This helps identify potential data quality issues early.
3.  **Prepare Data for Merging**: Standardizes `transaction_id` to string type, converts `transaction_date` to datetime objects (coercing errors to `NaT`), and `amount_usd` to numeric (filling any non-numeric with `0`). `status` and `payment_method` nulls are filled with 'NA' to ensure consistent data types across records.
4.  **Perform Full Outer Join**: A `pd.merge` with `how='outer'` is used on `transaction_id` to combine all records from both `ledger_df` and `gateway_df`. Suffixes `_ledger` and `_gateway` are added to differentiate columns from the original DataFrames.
5.  **Identify and Classify Discrepancies**: Filters the `merged_df` to create separate DataFrames for `missing_in_gateway`, `missing_in_ledger`, `amount_mismatches`, and `status_mismatches`. Each is then saved to its respective CSV file.
6.  **Build Final Reconciliation Report**: This crucial step assigns a `mismatch_type` to each discrepant row in the `merged_df` based on a defined priority (Missing in Ledger > Missing in Gateway > Amount Mismatch > Status Mismatch). It then selects and renames columns to match the required output schema (`transaction_id`, `transaction_date`, `merchant_id`, `ledger_amount`, `gateway_amount`, `ledger_status`, `gateway_status`, `mismatch_type`). Missing `transaction_date` are formatted to 'NA', and `merchant_id` ensures string type. Finally, it saves the `reconciliation_report_df` to `reconciliation_report.csv`.
7.  **Generate Summary Metrics**: Calculates counts for each type of discrepancy and the total number of discrepancies in the final report. These metrics are then saved into `summary_metrics.json`.

### Sample Output Structure of `reconciliation_report.csv`:

```csv
transaction_id,transaction_date,merchant_id,ledger_amount,gateway_amount,ledger_status,gateway_status,mismatch_type
transaction_0001,2023-01-01,merchant_A,100.50,0.00,success,NA,Missing in Gateway
transaction_0002,2023-01-02,merchant_B,0.00,200.75,NA,pending,Missing in Ledger
transaction_0003,2023-01-03,merchant_C,50.00,55.00,success,success,Amount Mismatch
transaction_0004,2023-01-04,merchant_D,120.00,120.00,success,failed,Status Mismatch
transaction_0005,2023-01-05,merchant_E,30.00,0.00,cancelled,NA,Missing in Gateway
transaction_0006,2023-01-06,merchant_F,0.00,75.00,NA,success,Missing in Ledger
```


